# Groups Generated by Translations
## From a single step to a lattice — the simplest geometry with a group behind it

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eduenez/erlangen-program/blob/main/notebooks/lattice-groups.ipynb)
[![License: CC BY 4.0](https://img.shields.io/badge/License-CC%20BY%204.0-lightgrey.svg)](https://creativecommons.org/licenses/by/4.0/)

---

A geometric transformation group is a set of transformations of a space
that forms a group under composition — closed, with an identity and
inverses. The simplest interesting examples come from translations of the
plane, and they're already rich enough to show the central idea of this
whole lesson series: **a shape's orbit under a group reveals the group's
own structure**, before any abstract definition does.

1. **One generator** — repeatedly applying a single translation gives an
   infinite cyclic group, and its orbit is a single line of copies.
2. **Two generators** — two independent translations generate a
   **lattice**: every integer combination $mv_1 + nv_2$. The same abstract
   group ($\mathbb{Z} \times \mathbb{Z}$) can look like a square grid, a
   hexagonal grid, or a slanted mess, depending only on the choice of
   generators.
3. **An interactive explorer** — sliders and dropdowns for both cases,
   including square and hexagonal presets.

Nothing here assumes more than "a translation moves every point by the
same fixed vector." Tinker freely — *Runtime → Run all* always resets you
to a clean state.


---
# 0. Setup

This notebook needs a live Python kernel for the sliders and dropdowns to
respond (Google Colab, or a local Jupyter/JupyterLab install) — a static
preview (e.g. viewing the raw file on GitHub) will show the code but not
let you interact with it.


In [1]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display


---
# 1. One generator: a discrete cyclic group

Fix a vector $v$ and consider the transformation $T(z) = z + v$. Applying
$T$ repeatedly — $T, T \circ T, T \circ T \circ T, \ldots$ — and also its
inverse $T^{-1}(z) = z - v$, generates the group
$\{\ldots, T^{-2}, T^{-1}, \mathrm{id}, T, T^2, \ldots\}$, an infinite
**cyclic group**. Every element is $z \mapsto z + kv$ for some integer $k$.
The orbit of a single shape under this group is a single-file line of
identical copies, spaced $|v|$ apart, all pointing the same way (a
translation never rotates anything).


---
# 2. Two generators: a lattice

Fix two vectors $v_1, v_2$ that don't point along the same line. The group
generated by translation-by-$v_1$ and translation-by-$v_2$ consists of
every transformation $z \mapsto z + mv_1 + nv_2$ for integers $m, n$ — as
an abstract group, always $\mathbb{Z} \times \mathbb{Z}$, regardless of
which $v_1, v_2$ you picked. But the *orbit* of a shape — the actual
picture in the plane — depends entirely on that choice: the same group can
tile the plane as a square grid, a hexagonal grid, or a slanted
parallelogram grid.


---
# 3. Interactive explorer


In [2]:
MOTIF = np.array([(0, 0), (0.32, 0), (0.16, 0.30), (0, 0)])  # a small flag, so
                                                              # orientation is visible

def vec(length, angle_deg):
    a = np.radians(angle_deg)
    return (length * np.cos(a), length * np.sin(a))

def cyclic_points(v, nrange):
    return [(k * v[0], k * v[1]) for k in range(-nrange, nrange + 1)]

def lattice_points(v1, v2, nrange):
    return [(m * v1[0] + n * v2[0], m * v1[1] + n * v2[1])
            for m in range(-nrange, nrange + 1) for n in range(-nrange, nrange + 1)]

def motif_trace(points, color):
    xs, ys = [], []
    for (px, py) in points:
        xs.extend(MOTIF[:, 0] + px); xs.append(None)
        ys.extend(MOTIF[:, 1] + py); ys.append(None)
    return go.Scatter(x=xs, y=ys, mode="lines", fill="toself",
                       line=dict(color=color), fillcolor=color)


In [3]:
mode = widgets.Dropdown(
    options=[("One generator — a cyclic group", "one"), ("Two generators — a lattice", "two")],
    value="two", description="Generators:", style={"description_width": "initial"})
preset = widgets.Dropdown(
    options=[("Square", "square"), ("Hexagonal", "hexagonal"), ("Custom", "custom")],
    value="square", description="Preset:", style={"description_width": "initial"})
v1_len = widgets.FloatSlider(min=0.3, max=2.5, step=0.1, value=1.3, description="|v1|:")
v1_ang = widgets.FloatSlider(min=0, max=360, step=5, value=30, description="angle(v1)°:")
v2_len = widgets.FloatSlider(min=0.3, max=2.5, step=0.1, value=1.0, description="|v2|:")
v2_ang = widgets.FloatSlider(min=0, max=360, step=5, value=90, description="angle(v2)°:")
n_range = widgets.IntSlider(min=1, max=6, value=3, description="range N:")
out = widgets.Output()

def redraw(*_):
    v1 = vec(v1_len.value, v1_ang.value)
    if mode.value == "one":
        pts = cyclic_points(v1, n_range.value)
        title = f"Cyclic group generated by one translation ({len(pts)} copies)"
    else:
        if preset.value == "square":
            v1_use, v2_use = (1, 0), (0, 1)
        elif preset.value == "hexagonal":
            v1_use, v2_use = (1, 0), (0.5, 3**0.5 / 2)
        else:
            v1_use, v2_use = v1, vec(v2_len.value, v2_ang.value)
        pts = lattice_points(v1_use, v2_use, n_range.value)
        title = f"Lattice generated by two translations ({len(pts)} copies)"
    fig = go.Figure(motif_trace(pts, "steelblue"))
    fig.update_layout(title=title, yaxis=dict(scaleanchor="x", scaleratio=1),
                       width=520, height=520, showlegend=False)
    with out:
        out.clear_output(wait=True)
        fig.show()

def update_visibility(*_):
    is_two = (mode.value == "two")
    preset.layout.display = "" if is_two else "none"
    is_custom = is_two and preset.value == "custom"
    show_v1 = (mode.value == "one") or is_custom
    v1_len.layout.display = v1_ang.layout.display = "" if show_v1 else "none"
    v2_len.layout.display = v2_ang.layout.display = "" if is_custom else "none"
    redraw()

for w in (mode, preset):
    w.observe(update_visibility, names="value")
for w in (v1_len, v1_ang, v2_len, v2_ang, n_range):
    w.observe(redraw, names="value")

update_visibility()
display(widgets.VBox([mode, preset, v1_len, v1_ang, v2_len, v2_ang, n_range, out]))


**Try it:**

- With **Square** selected, increase the range `N` — the pattern always
  stays a perfect grid, no matter how far out you go, because *every*
  lattice point looks exactly like every other one.
- Switch to **Hexagonal**. Notice $v_2$ is at $60°$ to $v_1$ with the same
  length — that specific relationship is what makes the grid come out
  hexagonally packed rather than square.
- Switch to **Custom** and drag $v_1, v_2$ close to parallel. What happens
  to the picture as the angle between them approaches $0°$? (This is why
  the definition requires them to be linearly independent — a real
  geometric group, not a degenerate one.)


---
# 4. The same lattice, different generators

Here's a subtlety the explorer can't show directly: **different pairs
$(v_1, v_2)$ can generate the *same* lattice** — the same set of points,
just reached by different combinations. For instance, $(v_1, v_2)$ and
$(v_1 + v_2, v_2)$ always generate the same lattice, because
$m v_1 + n v_2 = m(v_1+v_2) + (n-m) v_2$ — every point reachable with one
pair is reachable with the other.


In [4]:
def lattice_point_set(v1, v2, nrange, decimals=6):
    return {(round(p[0], decimals), round(p[1], decimals))
            for p in lattice_points(v1, v2, nrange)}

v1, v2 = (1.0, 0.0), (0.3, 0.9)
# (v1+v2, v2) reaches m'*v1 + (m'+n')*v2, so checking it against pair_a needs a
# wider range on pair_a (up to 2x) to avoid a false "missing point" from truncation.
pair_b_range = 3
pair_a = lattice_point_set(v1, v2, 2 * pair_b_range)
pair_b = lattice_point_set((v1[0] + v2[0], v1[1] + v2[1]), v2, pair_b_range)
# every point of pair_b should already be in pair_a (same lattice, just reached differently)
missing = pair_b - pair_a
print(f"points from the re-expressed generators not found in the original lattice: {len(missing)}")


points from the re-expressed generators not found in the original lattice: 0


---
# 5. Summary

- A single generator gives an infinite cyclic group; its orbit is one line
  of evenly spaced copies.
- Two independent generators give a lattice; the abstract group is always
  $\mathbb{Z} \times \mathbb{Z}$, but the geometric picture depends
  entirely on the specific vectors chosen — square and hexagonal grids are
  just two special, highly symmetric choices among infinitely many.
- Different generator pairs can produce the identical lattice — "the
  group" and "a set of generators for the group" are not the same thing,
  and this is the first place in the series where that distinction matters.


---
# 6. Exercises

1. **Degenerate case.** Set Custom $v_1, v_2$ to the exact same angle
   (parallel vectors). What does the "lattice" actually look like, and why
   does it fail to be 2-dimensional?
2. **Same lattice, found two ways.** Pick your own $v_1, v_2$, then predict
   (before checking) what $(2v_1 + v_2,\, v_1 + v_2)$ generates. Does it
   give the same lattice, a sublattice, or something else? Test it by
   extending Section 4's code.
3. **Density.** For the square lattice, how many lattice points lie within
   distance $R$ of the origin, as a function of $R$? (This is really
   asking how many integer pairs $(m,n)$ satisfy $m^2+n^2 \le R^2$ — no
   new code needed, just count in the existing point lists.)
4. **A third generator.** What if you had *three* vectors, no two
   parallel? Would the resulting group necessarily need all three to
   describe every element, or could two of them already generate
   everything reachable with all three? Give an example either way.


---
## License

This notebook is released under the [Creative Commons Attribution 4.0
International License (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/). You are free to copy, adapt,
and redistribute it, including for commercial purposes, provided you give
appropriate credit.

Suggested attribution: *"Groups Generated by Translations" by [Eduardo Dueñez](https://supernumero.us/about),
licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).*
